# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure that mlcroissant is installed in your environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
print("{}: {}".format(dataset.metadata.name, dataset.metadata.description))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema describes entities using `@id` fields. Let's list the available record sets and fields by their `@id`s.

In [ ]:
# Explore the dataset metadata structure
# Show available record sets and their @id

record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for record_set in record_sets:
        print("Record Set @id:", record_set['@id'])
        print("  Fields:")
        fields = record_set.get('field', [])
        for field in fields:
            print("    Field @id:", field['@id'], ", name:", field.get('name'))
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Entities are referenced by their `@id`.

Below, we extract all records for each available record set by their `@id`.

In [ ]:
# Collect record set @id values
record_sets_metadata = dataset.metadata.recordSet
record_set_ids = []

for record_set in record_sets_metadata:
    record_set_ids.append(record_set['@id'])
print("Record sets found:", record_set_ids)

# Load records for each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id} with shape {df.shape}")
    else:
        print(f"No records found for record set {record_set_id}")

# For illustration, view columns and preview of the first record set
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    if first_record_set_id in dataframes:
        print(f"Columns for {first_record_set_id}:", dataframes[first_record_set_id].columns.tolist())
        display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing numeric fields, and grouping data. Reference fields using their `@id`.

Below, we filter records, normalize a numeric field, and group the data by a categorical (demographic) field.

In [ ]:
# Example: EDA on GAD-7 score field
# Find numeric fields by @id for the first record set
example_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(example_record_set_id)

if df is not None:
    # Try to identify numeric fields (e.g., GAD-7 scores, PHQ-9 scores)
    # Use the field @id from the metadata
    numeric_field_id = None
    group_field_id = None
    for record_set in record_sets_metadata:
        if record_set['@id'] == example_record_set_id:
            for field in record_set.get('field', []):
                # Assume GAD-7 score field is present (for demonstration)
                if 'GAD-7' in str(field.get('name', '')).upper() or 'GAD7' in str(field.get('name', '')).upper():
                    numeric_field_id = field['@id'] if field['@id'] in df.columns else field.get('name')
                # Choose a demographic/group field (e.g., gender, if present)
                if 'gender' in str(field.get('name', '')).lower() or 'sex' in str(field.get('name', '')).lower():
                    group_field_id = field['@id'] if field['@id'] in df.columns else field.get('name')
    # If not found, fallback to column names
    if numeric_field_id is None:
        for col in df.columns:
            if 'gad' in col.lower():
                numeric_field_id = col
                break
    if group_field_id is None:
        for col in df.columns:
            if 'gender' in col.lower():
                group_field_id = col
                break
    print(f"Numeric field chosen: {numeric_field_id}")
    print(f"Grouping field chosen: {group_field_id}")

    # Filter records where GAD-7 score > threshold
    threshold = 10
    if numeric_field_id and numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize the GAD-7 score
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

        # Group by gender (or another demographic field)
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
            print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df)
        else:
            print(f"Group field '{group_field_id}' not available for grouping.")
    else:
        print(f"Numeric field '{numeric_field_id}' not found. Please adjust field selection.")
else:
    print("No records available in the primary record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot the distribution of GAD-7 scores and compare by gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of GAD-7 scores
if df is not None and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} scores")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Compare GAD-7 score mean by gender if group_field_id available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} Scores by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and processing the FAIR² Mental Health Survey Dataset with `mlcroissant`.

- Dataset fields and structure are referenced by their `@id` per Croissant standard.
- Data was loaded, filtered, normalized, and grouped for exploratory analysis.
- Visualizations show how mental health assessment scores may be distributed and vary across demographics.

Further analysis can include deeper statistical tests, handling missing values, or deploying machine learning models using this AI-ready data.